# 슬라이드 6 실시간 시연 — 괴테 표기 26종 → 한 사람

`queries_demo.py`의 Q1과 같은 질의입니다(`kt:P-goethe skos:altLabel`). 화면용으로 결과만 크게 보여 줍니다.

**순서**
1. 이 노트북을 `ktrans-schema.ttl`, `ktrans-data.ttl`과 **같은 폴더**에 둡니다. 저장소에서는 `ontology/`·`data/`의 두 파일을 이 폴더로 복사하세요.
2. 전날 밤: ① 준비 셀 → ▶ LIVE 셀을 실행해 "26 forms → 1 person"이 나오는지 확인하고, 결과 화면을 **스크린숏**으로 저장합니다(실패 대비).
3. 발표 당일: ① 준비 셀만 미리 실행 → ▶ LIVE 셀 출력은 지워 두고 그 셀이 화면 위쪽에 오게 스크롤 → 발표 중 Shift+Enter 한 번.

② 점검 셀은 리허설용입니다. 청중에게는 ▶ LIVE 셀만 보여 주세요.

In [ ]:
# ① 준비 셀 — 발표 전에 미리 실행 (청중에게 보여 줄 필요 없음)
import time, html
from pathlib import Path
from rdflib import Graph, Literal

from IPython.display import HTML, display

DATA_FILES = ["ktrans-schema.ttl", "ktrans-data.ttl"]   # queries_demo.py와 같은 파일

PREFIX = """
PREFIX ktrans: <https://ktrans.aks.ac.kr/ontology#>
PREFIX kt:     <https://ktrans.aks.ac.kr/id/>
PREFIX lrmoo:  <http://iflastandards.info/ns/lrm/lrmoo/>
PREFIX crm:    <http://www.cidoc-crm.org/cidoc-crm/7.1.3/>
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
"""

t0 = time.time()
g = Graph()
for f in DATA_FILES:
    if not Path(f).exists():
        raise FileNotFoundError(f"{f} 이 없습니다. 노트북을 ttl 파일과 같은 폴더로 옮기거나 경로를 고치세요.")
    g.parse(f, format="turtle")
print(f"{len(g):,} triples loaded in {time.time()-t0:.1f}s")   # 12,199 근처면 정상 (v3.1, 한국어·영어 메모)

def _name(person):
    rows = list(g.query(PREFIX + f"""
        SELECT ?n WHERE {{ <{person}> skos:prefLabel|rdfs:label ?n }}"""))
    names = {(r.n.language or ""): str(r.n) for r in rows}
    for lang in ("de", "en", "", "ko"):
        if lang in names:
            return names[lang]
    return next(iter(names.values()), "Goethe")

def show_result(res, person="https://ktrans.aks.ac.kr/id/P-goethe", expected=26):
    forms = sorted({str(r.form) for r in res})
    who = _name(person)
    n_works = len(list(g.query(PREFIX + "SELECT ?t WHERE { ?t a ktrans:Translation }")))
    chips = " · ".join(html.escape(f) for f in forms)
    page = f"""
    <div style="font-family:Arial,sans-serif;line-height:1.4">
      <div style="font-size:36px"><b>{len(forms)}</b> printed forms &nbsp;→&nbsp; <b>1</b> person:
        <span style="color:#6D2E46"><b>{html.escape(who)}</b></span></div>
      <div style="font-size:16px;color:#555;margin:8px 0 14px 0">{chips}</div>
      <div style="font-size:24px;color:#333">{n_works} translations, counted at once</div>
    </div>"""
    display(HTML(page))
    Path("live_demo_result.html").write_text(page, encoding="utf-8")   # 실패 대비용 사본
    if len(forms) != expected:
        print(f"[리허설 확인] 표기가 {len(forms)}개입니다(예상 {expected}). 슬라이드·대본 숫자와 맞는지 확인하세요.")


In [ ]:
# ② 점검 셀 — 리허설용. 발표 숫자와 데이터가 맞는지 확인
checks = {
    "번역본 (271)": "SELECT (COUNT(?t) AS ?n) WHERE { ?t a ktrans:Translation }",
    "괴테 표기 (26)": "SELECT (COUNT(?f) AS ?n) WHERE { kt:P-goethe skos:altLabel ?f }",
    "비평 진술 (160)": "SELECT (COUNT(?c) AS ?n) WHERE { ?c crm:P2_has_type kt:type-criticism }",
    "원문 문단 (183)": "SELECT (COUNT(?p) AS ?n) WHERE { ?p crm:P2_has_type kt:type-source-paragraph }",
}
for k, s in checks.items():
    print(f"{k:16s} → {int(list(g.query(PREFIX + s))[0][0])}")
print("※ 27번째 표기(작중인물 '베르테르')는 ttl에 없음 — 슬라이드 상자로만 말한다.")


In [ ]:
# ▶ LIVE — one author, many names
q = """
SELECT ?form WHERE {
  kt:P-goethe skos:altLabel ?form .
}
"""
show_result(g.query(PREFIX + q))
